# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

In [12]:
%matplotlib ipympl

In [13]:
import pandas as pd

from analysis.build_data import load_trace_df

In [14]:
run_name = "sweep-20261105-1621"

In [15]:
# one row per forecast of every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name)

## Expected calibration error

ECE (Naeini et al., 2015) is the average gap between confidence and accuracy, with forecasts binned by their stated confidence into $B$ bins:

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N}\,\bigl|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\bigr|$$

Here a forecast is the outcome prompts' yes/no call, `confidence` is the model's stated probability that the call is right, and a forecast is accurate when the call matches `outcome_occurred`.

In [ ]:
# scoreable forecasts: a yes/no call with a confidence, on a market that has resolved
calibrated = forecasts.dropna(
    subset=["predicted_outcome", "confidence", "outcome_occurred"]
)

In [ ]:
calibrated["correct"] = calibrated["predicted_outcome"] == calibrated[
    "outcome_occurred"
].astype(bool)

In [ ]:
B = 10

In [ ]:
confidence_bins = pd.IntervalIndex.from_breaks([b / B for b in range(B + 1)])

In [ ]:
calibrated["confidence_bin"] = pd.cut(
    calibrated["confidence"], confidence_bins, include_lowest=True
)

In [ ]:
def expected_calibration_error(forecasts: pd.DataFrame) -> float:
    """sum_b |B_b|/N * |acc(B_b) - conf(B_b)| over the confidence bins."""
    bins = forecasts.groupby("confidence_bin", observed=True)
    acc = bins["correct"].mean()
    conf = bins["confidence"].mean()
    weight = bins.size() / len(forecasts)
    return (weight * (acc - conf).abs()).sum()

In [ ]:
ece = expected_calibration_error(calibrated)

In [ ]:
ece_by_model = calibrated.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

In [ ]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = calibrated.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)

In [ ]:
reliability